# 🪞 Reflection Agents in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Generator and critic chains** - Two prompts, one model: a LinkedIn post writer and a content-strategist critic
2. **One reflection round by hand** - Feed the critique back as a message and watch the draft change
3. **The loop as a LangGraph graph** - Keep the whole exchange in `messages`, swapping roles so the critic reviews the writer's drafts
4. **Bounding and checking the loop** - Stop after a fixed number of drafts, and check each draft's length in code

## Where this fits
`modules/01_agents_from_scratch/notebooks/01_reflection_pattern.ipynb` built this generate → reflect loop from scratch with the raw OpenAI SDK, and module 09 (`modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`) replaced the free-text critique with structured, per-criterion grades. This notebook is the **LangGraph idiom** for the free-text version: the conversation itself is the state. `02_reflexion_agents.ipynb` then gives the critic external evidence to work with.

## Prerequisites
- `modules/01_agents_from_scratch/notebooks/01_reflection_pattern.ipynb` (the pattern without a framework)
- `StateGraph` and `add_messages` - `modules/03_react_agents/notebooks/01_react_agent_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📖 Part 1: What Is a Reflection Agent?

A **Reflection Agent** is an AI system that goes beyond just generating answers — it actively critiques its own outputs and keeps refining them. You can think of it as a loop between two roles:

- **Generator:** Drafts the initial response.
- **Reflector:** Reviews that draft, points out flaws or gaps, and suggests improvements.

This back-and-forth runs for a few iterations, and with each cycle, the output gets more polished, reliable, and useful. The AI essentially learns from its own mistakes in real time, much like a writer rewriting their draft after a round of editorial feedback.

| Property | Value |
|---|---|
| Origin | Madaan et al., *Self-Refine: Iterative Refinement with Self-Feedback*, NeurIPS 2023 ([arXiv:2303.17651](https://arxiv.org/abs/2303.17651)) |

---
## 📦 Step 1: Environment Setup

Load the API key and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Writer and critic
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

# temperature 0.7: each revision should be free to change wording, not repeat the last draft
writer_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)

print(f"✅ Critic: {llm.model_name} (temperature={llm.temperature}), writer: {writer_llm.model_name} (temperature={writer_llm.temperature})")

---
## ✍️ Part 2: Generator and Critic Chains

### 2.1 The Generator

The first component is a **LinkedIn post generator**. It drafts posts that can later be refined through self-review. The prompt has two parts:

- **System message:** the assistant's role as a professional LinkedIn content creator, responsible for generating *and revising* posts.
- **`MessagesPlaceholder`:** a slot filled at runtime with the conversation so far - the request, and later the drafts and critiques.

The prompt is piped (`|`) into the model to form a chain.

In [ ]:
# ============================================================================
# GENERATOR CHAIN: Prompt | writer model
# ============================================================================
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

post_creation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert LinkedIn content creator tasked with crafting compelling, professional, and high-performing LinkedIn posts. "
            "Create the most effective LinkedIn post possible based on the user's requirements. "
            "If the user provides feedback or suggestions, respond with an improved version that incorporates their input while enhancing overall quality and engagement. "
            "Reply with the post text only - no preamble, no commentary.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

linkedin_post_generator = post_creation_prompt | writer_llm


def word_count(text: str) -> int:
    return len(text.split())


post_request = HumanMessage(content="Create a LinkedIn post on AI tools for developers under 180 words.")
generated_post = linkedin_post_generator.invoke({"messages": [post_request]}).content

print(f"=== INITIAL LINKEDIN POST ({word_count(generated_post)} words) ===")
print(generated_post)

### 2.2 The Critic

The **critic** is a social-media strategist. It reviews the post for content quality, engagement potential and technical elements (hashtags, call to action, formatting), and returns free-text, actionable feedback.

In [ ]:
# ============================================================================
# CRITIC CHAIN: Prompt | critic model
# ============================================================================
social_media_critique_prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        """You are a LinkedIn content strategist and thought leadership expert. Analyze the given LinkedIn post and provide a comprehensive critique focusing on:

        **Content Quality & Professionalism:**
        - Overall quality, tone clarity, and LinkedIn best practices alignment
        - Structure, readability, and professional credibility building
        - Industry relevance and audience targeting
        
        **Engagement & Algorithm Optimization:**
        - Hook effectiveness and storytelling quality
        - Engagement potential (likes, comments, shares)
        - LinkedIn algorithm optimization factors
        - Word count and formatting effectiveness
        
        **Technical Elements:**
        - Hashtag relevance, reach, and strategic placement
        - Call-to-action strength and clarity
        - Use of formatting (line breaks, bullet points, mentions)
        
        Provide specific, actionable feedback that includes:
        - Key strengths and improvement areas
        - Concrete suggestions for enhancing engagement and professionalism
        - Practical recommendations for the next revision

        Keep your critique constructive and focused on measurable improvements, prioritizing actionable insights that will guide the post's revision and lead to tangible content enhancements."""
    ),
    MessagesPlaceholder(variable_name="messages")
])

social_media_critic = social_media_critique_prompt | llm

feedback_result = social_media_critic.invoke({"messages": [post_request, HumanMessage(content=generated_post)]}).content

print("=== SOCIAL MEDIA STRATEGIST FEEDBACK ===")
print(feedback_result[:1500])

### 2.3 🔁 One Reflection Round by Hand

The reflection loop is just message bookkeeping: the generator sees its own post as an `AIMessage` and the critique as the next `HumanMessage`, so it answers the critique with a revised post.

In [ ]:
# ============================================================================
# ONE MANUAL ROUND: request -> draft (AI) -> critique (human) -> revision
# ============================================================================
refined_post = linkedin_post_generator.invoke(
    {"messages": [post_request, AIMessage(content=generated_post), HumanMessage(content=feedback_result)]}
).content

print(f"=== REFINED LINKEDIN POST ({word_count(refined_post)} words) ===")
print(refined_post)

This works for one round, but tracking messages by hand for several rounds quickly becomes cumbersome. LangGraph gives us state management, conditional logic to decide whether to continue, and the orchestration between the two roles.

---
## 🔄 Part 3: The Loop as a LangGraph Graph

### Key Concepts:
- **`ContentState`**: one `messages` list with the `add_messages` reducer - drafts and critiques accumulate in order.
- **Role swap**: the critic must see the *writer's* drafts as the user's input. `social_critique_node` swaps AI ↔ human roles before calling the critic, and returns its feedback as a `HumanMessage` so the writer treats it as the user's reply.
- **A bounded loop**: `should_continue_refining` counts the drafts written so far and ends after `MAX_DRAFTS`.

In [ ]:
# ============================================================================
# GRAPH: create_post -> (social_critique -> create_post)* -> END
# ============================================================================
from typing import Annotated

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

MAX_DRAFTS = 3  # the first draft + 2 revisions


class ContentState(TypedDict):
    messages: Annotated[list, add_messages]


def post_creation_node(state: ContentState) -> dict:
    """Generate or improve the LinkedIn post based on the conversation so far."""
    return {"messages": [linkedin_post_generator.invoke({"messages": state["messages"]})]}


def social_critique_node(state: ContentState) -> dict:
    """Critique the latest post. The critic sees the writer's messages as the user's input."""
    message_role_map = {"ai": HumanMessage, "human": AIMessage}
    transformed_messages = [state["messages"][0]] + [
        message_role_map[msg.type](https://github.com/Sourav692/ai-engineering-demystified-06-agent-fundamentals/tree/main/modules/11_reflection/notebooks/content=msg.content) for msg in state["messages"][1:]
    ]
    strategy_feedback = social_media_critic.invoke({"messages": transformed_messages})
    return {"messages": [HumanMessage(content=strategy_feedback.content)]}


def should_continue_refining(state: ContentState):
    """Pure router: stop once MAX_DRAFTS posts have been written."""
    drafts = sum(1 for m in state["messages"] if m.type == "ai")
    return END if drafts >= MAX_DRAFTS else "social_critique"


content_workflow_builder = StateGraph(ContentState)
content_workflow_builder.add_node("create_post", post_creation_node)
content_workflow_builder.add_node("social_critique", social_critique_node)
content_workflow_builder.add_edge(START, "create_post")
content_workflow_builder.add_conditional_edges("create_post", should_continue_refining, ["social_critique", END])
content_workflow_builder.add_edge("social_critique", "create_post")

linkedin_workflow = content_workflow_builder.compile(checkpointer=InMemorySaver())
print("🧩 Nodes:", [n for n in linkedin_workflow.get_graph().nodes if not n.startswith("__")])

---
## 🧪 Part 4: Running and Checking the Loop

Stream the graph with `stream_mode="updates"` so each step prints once, as it happens. The checkpointer keeps the final state under the thread id, so we can read it back afterwards with `get_state`.

In [ ]:
# ============================================================================
# RUN: Stream each node's update
# ============================================================================
session_config = {"configurable": {"thread_id": "user1"}}
content_brief = HumanMessage(content="Create a LinkedIn post on AI tools for developers under 180 words.")

for update in linkedin_workflow.stream({"messages": [content_brief]}, session_config, stream_mode="updates"):
    for node, value in update.items():
        text = value["messages"][-1].content
        label = f"✍️  {node} ({word_count(text)} words)" if node == "create_post" else f"🧐 {node}"
        print(f"{label}: {text[:200].strip()}...\n")

The prompt asked for **under 180 words**. Word counts are something code checks better than any critic, so check every draft here rather than trusting the critique to notice:

In [ ]:
# ============================================================================
# CHECK EVERY DRAFT IN CODE: word counts from the checkpointed final state
# ============================================================================
final_state = linkedin_workflow.get_state(session_config)
drafts = [m.content for m in final_state.values["messages"] if m.type == "ai"]

print(f"Total messages in conversation: {len(final_state.values['messages'])}")
for i, draft in enumerate(drafts, start=1):
    n = word_count(draft)
    print(f"Draft {i}: {n} words {'✅' if n < 180 else '❌ over the limit'}")

print("\n=== FINAL POST ===")
print(drafts[-1])

Read the counts across drafts. Nothing in this loop checks the limit: the critic comments on hooks, hashtags and calls to action, and the writer adds whatever it asks for. Whether a revision drifts past 180 words varies from run to run - in an earlier version of this notebook, whose writer also opened every revision with "Thank you for the critique!", drafts 2 and 3 reached 251 and 282 words. The loop cannot know it broke a rule nobody checks; that is why module 09 moved checkable criteria (like length) into code.

Look also for **numbers the critique asked for**: when the critic says "add a statistic", the writer tends to supply one ("up to 30% more productive") with no source. Self-reflection cannot check facts it has no access to - that is exactly what `02_reflexion_agents.ipynb` adds.

To experiment, change `content_brief` - for example *"Write a LinkedIn post about remote work productivity tips under 150 words."* - and re-run Part 4.

---
## 📝 Summary

In this notebook, we learned:

### 1. Generator and critic chains
- One model, two prompts: a writer that drafts and revises, and a critic that returns actionable free-text feedback.

### 2. One reflection round by hand
- Reflection is message bookkeeping: the draft goes back as an `AIMessage`, the critique as the next `HumanMessage`.

### 3. The loop as a LangGraph graph
- With `add_messages`, the conversation is the state. The critic node swaps roles so it reviews the writer's drafts as user input.

### 4. Bounding and checking the loop
- A pure router stops after `MAX_DRAFTS`; a checkpointer lets you read the final state back.
- Self-reflection gives the model more attempts, but gains are not guaranteed. Check hard constraints in code - and for stronger results, give the critic external evidence (next notebook).

### Next Steps
- **`02_reflexion_agents.ipynb`**: Reflexion - a self-critique that names what is missing, web searches to fill the gaps, and a cited revision